In [ ]:
%cd /content

# 1) MotionBERT's code: only download it if the folder isn't there yet
![ -d MotionBERT ] || git clone -q https://github.com/Walter0807/MotionBERT.git

# 2) MotionBERT's starting weights (162 MB): only if the file isn't there yet
!mkdir -p MotionBERT/checkpoint/pose3d/FT_MB_release_MB_ft_h36m
![ -f MotionBERT/checkpoint/pose3d/FT_MB_release_MB_ft_h36m/best_epoch.bin ] || curl -sL -o MotionBERT/checkpoint/pose3d/FT_MB_release_MB_ft_h36m/best_epoch.bin https://huggingface.co/walterzhu/MotionBERT/resolve/main/checkpoint/pose3d/FT_MB_release_MB_ft_h36m/best_epoch.bin

# 3) Sonny's public data bundle (398 MB): only if it isn't there yet
![ -d Mocap3D_preprocessed_1118 ] || (curl -sL -o data.zip https://github.com/hyeon03-sketch/AI-application/releases/download/data-1118/Mocap3D_preprocessed_1118.zip && unzip -q data.zip)

# checks
!ls -lh MotionBERT/checkpoint/pose3d/FT_MB_release_MB_ft_h36m/
!ls
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content
total 163M
-rw-r--r-- 1 root root 163M Nov 17 14:45 best_epoch.bin
data.zip  Mocap3D_preprocessed_1118  MotionBERT  sample_data
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
# ---------- memory setting (must come before importing torch) ----------
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---------- setup ----------
import sys, json, time
import numpy as np
import torch
from functools import partial

MB   = "/content/MotionBERT"
DATA = "/content/Mocap3D_preprocessed_1118"
sys.path.insert(0, MB); sys.path.insert(0, DATA)
from lib.model.DSTformer import DSTformer
from compute_metrics import mpjpe, pa_mpjpe

def load_motionbert():
    m = DSTformer(dim_in=3, dim_out=3, dim_feat=512, dim_rep=512, depth=5, num_heads=8, mlp_ratio=2,
                  norm_layer=partial(torch.nn.LayerNorm, eps=1e-6), maxlen=243, num_joints=17)
    sd = torch.load(os.path.join(MB, "checkpoint/pose3d/FT_MB_release_MB_ft_h36m/best_epoch.bin"),
                    map_location="cpu", weights_only=True)["model_pos"]
    m.load_state_dict({k.replace("module.", "", 1): v for k, v in sd.items()}, strict=True)
    return m

sp = json.load(open(os.path.join(DATA, "splits.json")))
L, STRIDE, SCALING = 243, 81, "crop"

def normalize(x, w, h):
    out = x.copy(); out[..., 0] = x[..., 0] / w * 2 - 1; out[..., 1] = x[..., 1] / w * 2 - h / w; return out

def crop_scale(m):
    lo, hi = m[..., :2].reshape(-1, 2).min(0), m[..., :2].reshape(-1, 2).max(0); s = (hi - lo).max()
    out = m.copy(); out[..., :2] = ((m[..., :2] - (lo + hi - s) / 2) / s - 0.5) * 2; return out

def image_size(dataset, rid):
    if dataset == "rehab24":
        return (1920, 1080) if rid.endswith("c17") else (1080, 1920)
    return sp[dataset]["image_size_wh"]

def scale_input(dataset, rid, xx):
    return crop_scale(xx) if SCALING == "crop" else normalize(xx, *image_size(dataset, rid))

def make_clips(x, y):
    T = len(x)
    starts = list(range(0, T - L + 1, STRIDE)) if T >= L else [0]
    if T >= L and starts[-1] != T - L:
        starts.append(T - L)
    out = []
    for s in starts:
        idx = np.arange(s, s + L); mask = idx < T; idx = np.minimum(idx, T - 1)
        out.append((x[idx], y[idx], mask))
    return out

def build(split):
    X, Y, M = [], [], []
    for d in ("aspset", "rehab24"):
        K = np.load(os.path.join(DATA, f"{d}_kp2d_h36m.npz")); G = np.load(os.path.join(DATA, f"{d}_gt3d.npz"))
        for rid in sp[d][split]:
            for cx, cy, cm in make_clips(scale_input(d, rid, K[rid]), G[rid] / 1000.0):
                X.append(cx); Y.append(cy); M.append(cm)
    return np.stack(X).astype(np.float32), np.stack(Y).astype(np.float32), np.stack(M)

def masked_losses(pred, gt, mask):
    m = mask.float(); n = m.sum().clamp(min=1)
    pos = (torch.norm(pred - gt, dim=-1).mean(-1) * m).sum() / n
    s = (gt * pred).sum((-1, -2), keepdim=True) / (pred ** 2).sum((-1, -2), keepdim=True).clamp(min=1e-8)
    scale = (torch.norm(s * pred - gt, dim=-1).mean(-1) * m).sum() / n
    vm = m[:, 1:] * m[:, :-1]
    dv = (pred[:, 1:] - pred[:, :-1]) - (gt[:, 1:] - gt[:, :-1])
    vel = (torch.norm(dv, dim=-1).mean(-1) * vm).sum() / vm.sum().clamp(min=1)
    return pos + 0.5 * scale + 20 * vel, pos

Xtr, Ytr, Mtr = build("train")
Xva, Yva, Mva = build("val")
print("setup done:", Xtr.shape[0], "train clips,", Xva.shape[0], "val clips")

# ---------- small test on the GPU: 4 clips x 8 small steps, half precision ----------
dev = "cuda"
model = load_motionbert().to(dev).train()
opt = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")
torch.cuda.reset_peak_memory_stats()

BATCH, ACCUM = 4, 8
opt.zero_grad()
for step in range(ACCUM):
    i = np.random.choice(len(Xtr), BATCH, replace=False)
    xb, yb, mb = (torch.from_numpy(a[i]).to(dev) for a in (Xtr, Ytr, Mtr))
    t0 = time.time()
    with torch.autocast("cuda", dtype=torch.float16):
        pred = model(xb)
    pred = pred.float(); pred = pred - pred[:, :, :1]
    loss, pos = masked_losses(pred, yb, mb)
    scaler.scale(loss / ACCUM).backward()
    torch.cuda.synchronize()
    print(f"small step {step}: position error {pos.item() * 1000:.1f} mm   time {time.time() - t0:.2f} s")

scaler.step(opt); scaler.update(); opt.zero_grad()
print("peak GPU memory:", round(torch.cuda.max_memory_allocated() / 1e9, 1), "GB of 15")

setup done: 4534 train clips, 842 val clips
small step 0: position error 204.7 mm   time 1.00 s
small step 1: position error 154.8 mm   time 0.53 s
small step 2: position error 213.6 mm   time 0.53 s
small step 3: position error 173.3 mm   time 0.53 s
small step 4: position error 217.5 mm   time 0.53 s
small step 5: position error 152.0 mm   time 0.53 s
small step 6: position error 192.6 mm   time 0.53 s
small step 7: position error 173.3 mm   time 0.53 s
peak GPU memory: 7.7 GB of 15


In [ ]:
# ===== Cell 3: full fine-tuning, saved to Google Drive after every epoch =====
from google.colab import drive
drive.mount("/content/drive")                          # connect your Google Drive (allow the pop-up)
OUT = "/content/drive/MyDrive/motionbert_finetune"     # folder in your Drive for the saved models
os.makedirs(OUT, exist_ok=True)                        # create the folder if it doesn't exist

# free the GPU memory that the small test in Cell 2 used
for name in ("model", "opt", "scaler", "pred", "loss", "pos", "xb", "yb", "mb"):
    globals().pop(name, None)                          # delete the variable if it exists
torch.cuda.empty_cache()                               # give the freed memory back to the GPU

EPOCHS, BATCH, ACCUM = 20, 4, 8                        # 20 passes over the data; 4 clips x 8 small steps = 32 clips per update
torch.manual_seed(0); np.random.seed(0)                # seed 0 (required by the task sheet) so results can be repeated

model  = load_motionbert().to(dev)                     # fresh copy of the pretrained MotionBERT on the GPU
opt    = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)   # optimizer from the task sheet
sched  = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.99)             # learning rate shrinks 1% each epoch (MotionBERT's own setting)
scaler = torch.amp.GradScaler("cuda")                  # needed for half-precision (fp16) training

@torch.no_grad()                                       # no learning here, only measuring
def val_mpjpe():
    """Average 3D joint error (mm) on the validation clips."""
    model.eval()                                       # switch to measuring mode
    tot, n = 0.0, 0.0
    for s in range(0, len(Xva), 16):                   # 16 clips at a time
        xb = torch.from_numpy(Xva[s:s+16]).to(dev)                 # 2D input
        yb = torch.from_numpy(Yva[s:s+16]).to(dev)                 # true 3D (metres)
        mb = torch.from_numpy(Mva[s:s+16]).to(dev).float()         # 1 = real frame, 0 = padding
        with torch.autocast("cuda", dtype=torch.float16):
            pred = model(xb)                                       # predicted 3D
        pred = pred.float(); pred = pred - pred[:, :, :1]          # put the pelvis at 0, like the ground truth
        err = torch.norm(pred - yb, dim=-1).mean(-1)               # error per frame, averaged over the 17 joints
        tot += (err * mb).sum().item(); n += mb.sum().item()       # skip padded frames
    model.train()                                      # back to training mode
    return tot / n * 1000                              # metres -> mm

# ---- continue from Drive if an earlier run was cut off ----
last_path = os.path.join(OUT, "last.pt")
if os.path.exists(last_path):
    ck = torch.load(last_path, map_location=dev, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    start, best, hist = ck["epoch"] + 1, ck["best"], ck["hist"]
    print(f"continuing from epoch {start}, best val so far {best:.1f} mm")
else:
    start, best = 0, float("inf")
    before = val_mpjpe()                               # validation error BEFORE fine-tuning (for the before/after table)
    hist = [{"epoch": "before", "val_mpjpe_mm": round(before, 2)}]
    print(f"before fine-tuning: val MPJPE {before:.1f} mm")

# ---- training ----
model.train()
for ep in range(start, EPOCHS):
    t0 = time.time()
    order = np.random.default_rng(ep).permutation(len(Xtr))   # shuffle the clips (same shuffle if rerun)
    nsteps = len(order) // BATCH
    run = 0.0                                                 # running total of the training error
    opt.zero_grad()
    for k in range(nsteps):
        i = order[k*BATCH:(k+1)*BATCH]                        # pick the next 4 clips
        xb, yb, mb = (torch.from_numpy(a[i]).to(dev) for a in (Xtr, Ytr, Mtr))
        with torch.autocast("cuda", dtype=torch.float16):
            pred = model(xb)
        pred = pred.float(); pred = pred - pred[:, :, :1]     # pelvis at 0
        loss, pos = masked_losses(pred, yb, mb)               # position + 0.5 scale + 20 velocity
        scaler.scale(loss / ACCUM).backward()                 # collect the gradients
        run += pos.item()
        if (k + 1) % ACCUM == 0 or k + 1 == nsteps:           # every 8 small steps: update the weights once
            scaler.step(opt); scaler.update(); opt.zero_grad()
    sched.step()                                              # lower the learning rate a bit

    v = val_mpjpe()                                           # check on validation data
    mins = (time.time() - t0) / 60
    hist.append({"epoch": ep, "train_mpjpe_mm": round(run / nsteps * 1000, 2),
                 "val_mpjpe_mm": round(v, 2), "minutes": round(mins, 1)})
    if v < best:                                              # new best epoch -> save the weights
        best = v
        torch.save(model.state_dict(), os.path.join(OUT, "best.pt"))
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epoch": ep, "best": best, "hist": hist}, last_path)
    json.dump(hist, open(os.path.join(OUT, "history.json"), "w"), indent=1)   # readable log
    print(f"epoch {ep:2d} | train {run/nsteps*1000:6.1f} mm | val {v:6.1f} mm | best {best:6.1f} | {mins:.1f} min")

print("done. best val MPJPE:", round(best, 1), "mm")

Mounted at /content/drive
before fine-tuning: val MPJPE 212.6 mm
epoch  0 | train   96.0 mm | val   86.6 mm | best   86.6 | 12.7 min
epoch  1 | train   58.4 mm | val   81.8 mm | best   81.8 | 12.8 min
epoch  2 | train   52.4 mm | val   79.1 mm | best   79.1 | 12.7 min
epoch  3 | train   48.0 mm | val   74.8 mm | best   74.8 | 12.8 min
epoch  4 | train   44.6 mm | val   74.5 mm | best   74.5 | 12.8 min
epoch  5 | train   41.8 mm | val   73.9 mm | best   73.9 | 12.8 min
epoch  6 | train   39.0 mm | val   74.5 mm | best   73.9 | 12.8 min
epoch  7 | train   37.5 mm | val   72.8 mm | best   72.8 | 12.8 min
epoch  8 | train   35.6 mm | val   73.5 mm | best   72.8 | 12.8 min
epoch  9 | train   34.2 mm | val   74.5 mm | best   72.8 | 12.8 min
epoch 10 | train   32.9 mm | val   74.2 mm | best   72.8 | 12.8 min
epoch 11 | train   32.1 mm | val   71.8 mm | best   71.8 | 12.8 min
epoch 12 | train   31.1 mm | val   73.2 mm | best   71.8 | 12.8 min
epoch 13 | train   30.2 mm | val   73.8 mm | best  

In [ ]:
# ===== Cell 4: evaluation before and after fine-tuning =====
import subprocess
EVAL = os.path.join(OUT, "eval"); os.makedirs(EVAL, exist_ok=True)   # results folder in your Drive

# free the GPU memory from training
for name in ("opt", "scaler", "sched", "pred", "loss", "pos", "xb", "yb", "mb"):
    globals().pop(name, None)
torch.cuda.empty_cache()

@torch.no_grad()
def predict(net, kp):
    """One whole recording: (T,17,3) scaled 2D -> (T,17,3) 3D in mm, pelvis at 0."""
    T = len(kp)
    starts = list(range(0, max(T - L, 0) + 1, STRIDE))           # 243-frame windows, every 81 frames
    if T > L and starts[-1] != T - L:
        starts.append(T - L)                                     # last window ends exactly at the last frame
    out, cnt = np.zeros((T, 17, 3)), np.zeros(T)                 # sum of predictions and how many windows hit each frame
    for b in range(0, len(starts), 8):                           # 8 windows at a time
        idx = [np.arange(s, s + L) for s in starts[b:b + 8]]     # frame numbers of each window
        x = np.stack([kp[np.minimum(i, T - 1)] for i in idx])    # short recordings: repeat the last frame
        p = net(torch.from_numpy(x.astype(np.float32)).to(dev)).float()
        p = (p - p[:, :, :1]).cpu().numpy()                      # pelvis at 0
        for i, pi in zip(idx, p):
            real = i < T                                         # ignore padded frames
            out[i[real]] += pi[real]; cnt[i[real]] += 1
    return out / cnt[:, None, None] * 1000                       # average overlapping windows, metres -> mm

def run_eval(net, tag):
    """Predict ASPset test + VIDIMU, run the team's two scripts, return the two json results."""
    net.eval()
    K = np.load(os.path.join(DATA, "aspset_kp2d_h36m.npz"))
    asp = {rid: predict(net, scale_input("aspset", rid, K[rid])) for rid in sp["aspset"]["test"]}   # test ids ONLY
    K = np.load(os.path.join(DATA, "vidimu_kp2d_h36m.npz"))
    vid = {rid: predict(net, scale_input("vidimu", rid, K[rid])) for rid in K.files}
    a_pred, v_pred = os.path.join(EVAL, f"aspset_pred3d_{tag}.npz"), os.path.join(EVAL, f"vidimu_pred3d_{tag}.npz")
    np.savez_compressed(a_pred, **asp); np.savez_compressed(v_pred, **vid)
    a_out, v_out = os.path.join(EVAL, f"metrics_pos_{tag}.json"), os.path.join(EVAL, f"vidimu_metrics_{tag}.json")
    subprocess.run([sys.executable, os.path.join(DATA, "compute_metrics.py"), "--pred", a_pred,
                    "--gt", os.path.join(DATA, "aspset_gt3d.npz"), "--out", a_out], check=True, capture_output=True)
    subprocess.run([sys.executable, os.path.join(DATA, "vidimu_eval.py"), "--pred", v_pred,
                    "--imu", os.path.join(DATA, "vidimu_imu_angles.npz"), "--out", v_out], check=True, capture_output=True)
    print(f"{tag}: {len(asp)} ASPset test recordings, {len(vid)} VIDIMU recordings done")
    return json.load(open(a_out)), json.load(open(v_out))

# ---- before: the original pretrained MotionBERT ----
res = {"before": run_eval(load_motionbert().to(dev), "before")}

# ---- after: your fine-tuned best epoch ----
net = load_motionbert()
net.load_state_dict(torch.load(os.path.join(OUT, "best.pt"), map_location="cpu", weights_only=True))
res["after"] = run_eval(net.to(dev), "after")

# ---- section 6 table ----
rows = [("ASPset MPJPE (mm)",              lambda a, v: a["position"]["summary"]["mpjpe_mm"]["mean"]),
        ("ASPset PA-MPJPE (mm)",           lambda a, v: a["position"]["summary"]["pa_mpjpe_mm"]["mean"]),
        ("VIDIMU r, no walking",           lambda a, v: v["no_walking_A04_A13"]["r"]),
        ("VIDIMU mean-removed RMSE (deg), no walking", lambda a, v: v["no_walking_A04_A13"]["mean_removed_rmse_deg"]),
        ("VIDIMU r, all",                  lambda a, v: v["all_A01_A13"]["r"]),
        ("VIDIMU mean-removed RMSE (deg), all", lambda a, v: v["all_A01_A13"]["mean_removed_rmse_deg"]),
        ("ROM diff (deg), no walking",     lambda a, v: v["no_walking_A04_A13"]["rom_diff_deg"]),
        ("Rep count same (%), no walking", lambda a, v: v["no_walking_A04_A13"]["reps_same_pct"]),
        ("Walking L-R knee r (IMU -0.30)", lambda a, v: v["walking_left_right_knee_r"]["model"])]
print(f"\n{'metric':45s}{'before':>10s}{'after':>10s}")
for name, get in rows:
    b, a = get(*res["before"]), get(*res["after"])
    print(f"{name:45s}{b:10.2f}{a:10.2f}")
print("\nparameters (M):", round(sum(p.numel() for p in net.parameters()) / 1e6, 1))
print("training GPU hours (T4):", round(sum(h.get("minutes", 0) for h in hist) / 60, 2))
print("best epoch:", min((h for h in hist if h["epoch"] != "before"), key=lambda h: h["val_mpjpe_mm"])["epoch"])

before: 60 ASPset test recordings, 205 VIDIMU recordings done
after: 60 ASPset test recordings, 205 VIDIMU recordings done

metric                                           before     after
ASPset MPJPE (mm)                                179.29     73.51
ASPset PA-MPJPE (mm)                              72.12     42.92
VIDIMU r, no walking                               0.90      0.93
VIDIMU mean-removed RMSE (deg), no walking        11.01      7.58
VIDIMU r, all                                      0.87      0.90
VIDIMU mean-removed RMSE (deg), all                9.94      8.17
ROM diff (deg), no walking                        -6.32     -6.30
Rep count same (%), no walking                    66.20     71.90
Walking L-R knee r (IMU -0.30)                     0.06     -0.15

parameters (M): 42.5
training GPU hours (T4): 4.26
best epoch: 15
